# Strategy Pivot

**Last updated:** 2026-04-15

**Track:** Learning | **Sub-Ability:** Reinforcement / meta-control

Given a history of attempts at solving a problem, can the model learn *when to stop* — i.e. decide between:

- `CONTINUE` — stay the course, current strategy is working or still progressing
- `ADJUST_PARAMS` — keep the strategy but vary parameters; errors show progress
- `PIVOT` — abandon this strategy; it is stuck (same error, no state change)

This probes a common agent failure mode: over-committing to a broken approach (e.g. 133 JS-injection calls in one session, 50 calls fighting a TeX toolchain). The material teaches pivot-trigger signals explicitly; the test items require applying those signals to new histories.

**Difficulty grid:** complexity (clear / subtle / adversarial) × evidence (few / mid / many attempts) × 3 seeds = **27 items**.

Each (complexity, evidence) cell contains one CONTINUE, one ADJUST_PARAMS, and one PIVOT, so label-frequency shortcuts don't help.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import json
import time

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

VALID_DECISIONS = ['ADJUST_PARAMS', 'CONTINUE', 'PIVOT']  # order: compound label first

def extract_decision(response: str) -> str:
    """Extract one of CONTINUE / ADJUST_PARAMS / PIVOT from model response.

    Scans short last-lines first (the model is instructed to put the decision
    on the final line), then falls back to first match anywhere. We strip
    backticks/asterisks but NOT underscores, because ADJUST_PARAMS contains
    one. We also normalize 'adjust params' / 'adjust-params' to the canonical
    underscored form.
    """
    text = strip_thinking(response)
    text = re.sub(r'[`*]', '', text)
    upper = text.upper()
    upper = re.sub(r'\bADJUST[\s\-]+PARAMS\b', 'ADJUST_PARAMS', upper)
    lines = [l.strip() for l in upper.strip().split('\n') if l.strip()]

    # Last-line preference (short, decision-only lines)
    for line in reversed(lines):
        clean = line.strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?:')
        if len(clean) <= 40:
            for d in VALID_DECISIONS:
                if re.search(rf'(?<![A-Z_]){d}(?![A-Z_])', clean):
                    return d

    # Fall back to any mention in the full text
    for d in VALID_DECISIONS:
        if re.search(rf'(?<![A-Z_]){d}(?![A-Z_])', upper):
            return d
    return ''

In [ ]:
# Constants used in results analysis
COMPLEXITIES = ['clear_signal', 'subtle_signal', 'adversarial']
EVIDENCE_LEVELS = ['few', 'mid', 'many']
SEEDS = 3
DECISIONS = ['CONTINUE', 'ADJUST_PARAMS', 'PIVOT']

In [ ]:
# === Load dataset from Kaggle ===
import os, glob
candidates = glob.glob('/kaggle/input/**/strategy_pivot_dataset.csv', recursive=True)
if candidates:
    csv_path = candidates[0]
else:
    csv_path = '/kaggle/input/strategy_pivot_dataset.csv'
print(f'Loading from: {csv_path}')
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items')
print(dataset[['difficulty_label', 'expected']].value_counts().to_string())

In [ ]:
PRE_P = """You are overseeing an autonomous agent that is trying to solve a task. You must decide what the agent should do next.

The agent has three options:
  - CONTINUE: stay with the current strategy; it is working or still making progress.
  - ADJUST_PARAMS: keep the strategy, but vary its parameters; errors are changing and state is advancing.
  - PIVOT: abandon the current strategy; it is stuck (same error repeating, no state change) or a known alternative is available.

{test_input}

Reply with ONLY one word: CONTINUE, ADJUST_PARAMS, or PIVOT. Nothing else."""

STUDY_P = """You will be evaluated on your ability to decide when an autonomous agent should CONTINUE, ADJUST_PARAMS, or PIVOT.

Below are worked examples that illustrate the correct rules. Study them carefully and write concise notes that you can use on later items. Focus on the signals that distinguish the three decisions.

{material}

Write a clear set of notes (3-8 bullet points) that identify:
1. The trigger signals for PIVOT (what must be true in the history?).
2. The trigger signals for ADJUST_PARAMS.
3. The trigger signals for CONTINUE.
4. Common traps / adversarial patterns that should NOT change the decision.
"""

POST_P = """You previously studied the rules for deciding CONTINUE / ADJUST_PARAMS / PIVOT and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Now apply those rules to the following new history.

{test_input}

Reply with ONLY one word: CONTINUE, ADJUST_PARAMS, or PIVOT. Nothing else."""

## Evaluation

In [ ]:
available = list(kbench.llms.keys())
all_results = []

@kbench.task(name='strategy_pivot',
             description='Pre/post learning strategy pivot — decide CONTINUE / ADJUST_PARAMS / PIVOT from attempt history')
def strategy_pivot(llm, material: str, test_input: str, expected: str,
                   complexity: str, evidence: str, num_attempts: int,
                   difficulty_label: str, seed: int, item_desc: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    tid = f'{complexity}_{evidence}_s{seed}'

    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(test_input=test_input))
    pre_time = time.time() - t0
    pre_extracted = extract_decision(pre_raw)
    pre_correct = pre_extracted == expected

    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, test_input=test_input))
    post_time = time.time() - t0
    post_extracted = extract_decision(post_raw)
    post_correct = post_extracted == expected

    all_results.append({
        'timestamp': ts, 'model': model_name, 'task_id': tid,
        'complexity': complexity, 'evidence': evidence,
        'num_attempts': int(num_attempts), 'difficulty_label': difficulty_label,
        'seed': int(seed), 'item_desc': item_desc,
        'test_input': test_input, 'expected': expected,
        'pre_raw': pre_raw, 'pre_extracted': pre_extracted, 'pre_correct': int(pre_correct),
        'study_notes': notes,
        'post_raw': post_raw, 'post_extracted': post_extracted, 'post_correct': int(post_correct),
        'pre_time_s': pre_time, 'study_time_s': study_time, 'post_time_s': post_time
    })

    b = 'Y' if pre_correct else 'N'
    l = 'Y' if post_correct else 'N'
    print(f'[{model_name:40s}] [{difficulty_label:20s}] expected="{expected:14s}"  pre="{pre_extracted:14s}"({b})  post="{post_extracted:14s}"({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

try:
    runs = strategy_pivot.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                   n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.DataFrame(all_results)
print(f'Total runs: {len(results)}\n')

# Per-model summary
print('SCALING CHECK — Pre/Post accuracy by model:')
print('=' * 70)
for model in sorted(results['model'].unique()):
    sub = results[results['model'] == model]
    pre = sub['pre_correct'].mean()
    post = sub['post_correct'].mean()
    gain = post - pre
    print(f'  {str(model):45s}  pre={pre:.1%}  post={post:.1%}  gain={gain:+.1%}  (n={len(sub)})')

# Per model x complexity
print()
print('By model x complexity (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for c in COMPLEXITIES:
        sub = results[(results['model'] == model) & (results['complexity'] == c)]
        if len(sub) > 0:
            row += f'  {c}={sub["pre_correct"].mean():.0%}'
    print(row)

# Per model x evidence
print()
print('By model x evidence (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for ev in EVIDENCE_LEVELS:
        sub = results[(results['model'] == model) & (results['evidence'] == ev)]
        if len(sub) > 0:
            row += f'  {ev}={sub["pre_correct"].mean():.0%}'
    print(row)

# Per-decision breakdown (are models biased toward one label?)
print()
print('By model x expected-decision (PRE accuracy):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for d in DECISIONS:
        sub = results[(results['model'] == model) & (results['expected'] == d)]
        if len(sub) > 0:
            row += f'  {d}={sub["pre_correct"].mean():.0%}'
    print(row)

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Item: {row["item_desc"]}')
    print(f'Expected: "{row["expected"]}"')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {str(row["study_notes"])[:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Strategy Pivot: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('strategy_pivot_results.png', dpi=150, bbox_inches='tight')
plt.show()